In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
"""
=============================================================
  ConvNeXt-Tiny + Bidirectional Cross-Attention
  Multimodal Skin Cancer Classifier
  ✅ Uses pre-split train/val/test data
  ✅ Hyperparameters aligned with GNN+MaxViT reference
=============================================================
"""

# ─────────────────────────────────────────────
# 0.  Imports
# ─────────────────────────────────────────────
import os, re, warnings
import numpy as np
import pandas as pd
import cv2
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
from tqdm import tqdm

# ─────────────────────────────────────────────
# 1.  CFG  (aligned with GNN+MaxViT reference)
# ─────────────────────────────────────────────
class CFG:
    IMG_SIZE      = 224
    MEAN          = [0.485, 0.456, 0.406]
    STD           = [0.229, 0.224, 0.225]

    # ── Model dims ──────────────────────────
    D             = 256          # shared projection dimension
    ATTN_HEADS    = 4            # cross-attention heads  (D must be divisible)
    META_TOKENS   = 4            # number of meta tokens  (same as GNN)
    DROPOUT       = 0.50         # same as GNN

    # ── ConvNeXt freezing ───────────────────
    FREEZE_STAGES = 3            # freeze first 2 stages (mirrors GNN FREEZE_BLOCKS=2)

    # ── Training ────────────────────────────
    EPOCHS        = 25
    BATCH_SIZE    = 16
    GRAD_ACCUM    = 2
    NUM_WORKERS   = 2
    PATIENCE      = 10
    SEED          = 42

    # ── Loss ────────────────────────────────
    LABEL_SMOOTH  = 0.10
    FOCAL_ALPHA   = 0.25
    FOCAL_GAMMA   = 2.0

    # ── LR (same split as GNN) ───────────────
    LR            = 2e-5         # backbone
    HEAD_LR       = 2e-4         # projection + attention + classifier head
    WEIGHT_DECAY  = 0.05

    # ── TTA ─────────────────────────────────
    TTA_STEPS     = 5

    # ── Paths ────────────────────────────────
    BASE_DIR      = "/kaggle/input/datasets/mokamohamed/final-data-set"
    TARGET_COL    = "class"

# ─────────────────────────────────────────────
# 2.  Device & Seed
# ─────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(CFG.SEED)
np.random.seed(CFG.SEED)

# ─────────────────────────────────────────────
# 3.  Load Pre-Split Data
# ─────────────────────────────────────────────
def load_split(split_name):
    split_folder = os.path.join(CFG.BASE_DIR, split_name, split_name)
    csv_path     = os.path.join(split_folder, f"{split_name}.csv")
    images_dir   = os.path.join(split_folder, "images")
    df = pd.read_csv(csv_path)
    df["images_dir"] = images_dir
    return df

train_df = load_split("train")
val_df   = load_split("val")
test_df  = load_split("test")

classes      = sorted(train_df[CFG.TARGET_COL].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
NUM_CLASSES  = len(classes)
classes_str  = [str(c) for c in classes]

for df_ in [train_df, val_df, test_df]:
    df_["label"] = df_[CFG.TARGET_COL].map(class_to_idx)

print(f"Classes     : {classes_str}")
print(f"Num classes : {NUM_CLASSES}")
print(f"Train size  : {len(train_df)}")
print(f"Val size    : {len(val_df)}")
print(f"Test size   : {len(test_df)}")
print(f"\nTrain distribution:\n{train_df['label'].value_counts()}")

# ─────────────────────────────────────────────
# 4.  Image Name Fix  (identical to GNN)
# ─────────────────────────────────────────────
def normalize_name(fname):
    x    = str(fname).replace(".jpg", "")
    nums = re.findall(r'(\d{7})', x)
    if nums:
        return f"ISIC_{nums[-1]}.jpg"
    return None

for df_ in [train_df, val_df, test_df]:
    if "image_fixed" not in df_.columns:
        df_["image_fixed"] = df_["image"].apply(normalize_name)
    df_.dropna(subset=["image_fixed"], inplace=True)
    df_.reset_index(drop=True, inplace=True)

# ─────────────────────────────────────────────
# 5.  META_COLS  (identical to GNN)
# ─────────────────────────────────────────────
DROP_COLS = ["image", "isic_id", "patient_id", "year",
             "class", "image_fixed", "images_dir"]
META_COLS = [c for c in train_df.columns
             if c not in DROP_COLS + ["label"]]
META_DIM  = len(META_COLS)
print(f"\nMeta columns ({META_DIM}): {META_COLS}")

# ─────────────────────────────────────────────
# 6.  Augmentation  (identical to GNN)
# ─────────────────────────────────────────────
def get_train_transform(img_size=CFG.IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Rotate(limit=15, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05,
                           rotate_limit=10, p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.1,
                                   contrast_limit=0.1, p=0.4),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.Sharpen(alpha=(0.1, 0.2), lightness=(0.9, 1.0), p=1.0),
        ], p=0.1),
        A.Normalize(mean=CFG.MEAN, std=CFG.STD),
        ToTensorV2()
    ])

def get_val_transform(img_size=CFG.IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=CFG.MEAN, std=CFG.STD),
        ToTensorV2()
    ])

# ─────────────────────────────────────────────
# 7.  Dataset  (identical to GNN)
# ─────────────────────────────────────────────
class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(row["images_dir"], row["image_fixed"])
        img      = cv2.imread(img_path)

        if img is None:
            return self.__getitem__((idx + 1) % len(self.df))

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))

        if self.transform:
            img = self.transform(image=img)["image"]

        meta  = torch.tensor(row[META_COLS].values.astype(np.float32))
        label = torch.tensor(row["label"], dtype=torch.long)
        return img, meta, label

# ─────────────────────────────────────────────
# 8.  Focal Loss + Label Smoothing  (identical to GNN)
# ─────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, logits, targets):
        num_classes = logits.size(1)
        with torch.no_grad():
            smooth = torch.full_like(
                logits, CFG.LABEL_SMOOTH / (num_classes - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - CFG.LABEL_SMOOTH)
        log_prob = F.log_softmax(logits, dim=1)
        ce       = -(smooth * log_prob).sum(dim=1)
        pt       = torch.exp(-ce)
        loss     = CFG.FOCAL_ALPHA * (1 - pt) ** CFG.FOCAL_GAMMA * ce
        return loss.mean()

# ─────────────────────────────────────────────
# 9.  Bidirectional Cross-Attention Block
#
#  Two directions per layer:
#   (a) Image → Meta  : image queries attend to meta keys/values
#   (b) Meta  → Image : meta queries attend to image keys/values
#  Each direction is a standard Multi-Head Attention + residual + LayerNorm.
#  Stacked N_LAYERS=2 times.
# ─────────────────────────────────────────────
class BidirectionalCrossAttentionBlock(nn.Module):
    """
    One layer of bidirectional cross-attention between
    image tokens (shape: B, Ri, D) and meta tokens (shape: B, Rm, D).
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        # Image attends to Meta
        self.img_to_meta = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm_img    = nn.LayerNorm(d_model)

        # Meta attends to Image
        self.meta_to_img = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm_meta   = nn.LayerNorm(d_model)

        # Per-modality FFN  (2× expand, same as Transformer convention)
        self.ffn_img  = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model))
        self.norm_ffn_img  = nn.LayerNorm(d_model)

        self.ffn_meta = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model))
        self.norm_ffn_meta = nn.LayerNorm(d_model)

        self.drop = nn.Dropout(dropout)

    def forward(self, img_tokens, meta_tokens):
        # ── (a) image queries attend to meta ──────────────────────────────
        img_ca, _  = self.img_to_meta(
            query=img_tokens, key=meta_tokens, value=meta_tokens)
        img_tokens = self.norm_img(img_tokens + self.drop(img_ca))

        # ── (b) meta queries attend to image ──────────────────────────────
        meta_ca, _ = self.meta_to_img(
            query=meta_tokens, key=img_tokens, value=img_tokens)
        meta_tokens = self.norm_meta(meta_tokens + self.drop(meta_ca))

        # ── FFN for each modality ──────────────────────────────────────────
        img_tokens  = self.norm_ffn_img(
            img_tokens  + self.drop(self.ffn_img(img_tokens)))
        meta_tokens = self.norm_ffn_meta(
            meta_tokens + self.drop(self.ffn_meta(meta_tokens)))

        return img_tokens, meta_tokens


# ─────────────────────────────────────────────
# 10. Full Model
# ─────────────────────────────────────────────
class ConvNeXtBiCrossAttn(nn.Module):
    """
    ConvNeXt-Tiny backbone  +  Bidirectional Cross-Attention fusion
    with clinical metadata.

    Architecture flow:
      images  ──► ConvNeXt-Tiny (stages 0-3, stages 0-1 frozen)
                  ──► flatten spatial ──► Linear(C→D) ──► img_tokens  [B, Ri, D]
      metadata ──► Linear(META_DIM → D*META_TOKENS) ──► reshape
                  ──► meta_tokens  [B, META_TOKENS, D]

      [img_tokens, meta_tokens]
          ──► BidirCrossAttn × 2 layers
          ──► mean-pool img_tokens
          ──► Linear(D → D//2) ──► GELU ──► Dropout ──► Linear(D//2 → num_classes)
    """
    N_CROSS_LAYERS = 2   # number of stacked bidirectional cross-attn layers

    def __init__(self, meta_input_dim: int, num_classes: int):
        super().__init__()

        # ── Backbone ────────────────────────────────────────────────────────
        self.backbone = timm.create_model(
            "convnext_tiny",
            pretrained=True, num_classes=0, global_pool=""
        )
        # Freeze first FREEZE_STAGES stages (mirrors GNN FREEZE_BLOCKS)
        for i, stage in enumerate(self.backbone.stages):
            if i < CFG.FREEZE_STAGES:
                for param in stage.parameters():
                    param.requires_grad = False

        # Probe backbone output channels
        with torch.no_grad():
            dummy = torch.zeros(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)
            feat  = self.backbone(dummy)   # shape: (1, C, H, W)
        C      = feat.shape[1]
        self.R = feat.shape[2] * feat.shape[3]   # number of spatial tokens

        # ── Projection layers ────────────────────────────────────────────────
        self.img_proj = nn.Sequential(
            nn.Linear(C, CFG.D),
            nn.LayerNorm(CFG.D),
            nn.GELU()
        )

        self.meta_proj = nn.Sequential(
            nn.Linear(meta_input_dim, CFG.D * CFG.META_TOKENS),
            nn.LayerNorm(CFG.D * CFG.META_TOKENS),
            nn.GELU()
        )

        # ── Bidirectional Cross-Attention stack ──────────────────────────────
        self.cross_attn_layers = nn.ModuleList([
            BidirectionalCrossAttentionBlock(
                d_model=CFG.D,
                n_heads=CFG.ATTN_HEADS,
                dropout=CFG.DROPOUT
            )
            for _ in range(self.N_CROSS_LAYERS)
        ])

        # ── Classification head ──────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Linear(CFG.D, CFG.D // 2),
            nn.GELU(),
            nn.Dropout(CFG.DROPOUT),
            nn.Linear(CFG.D // 2, num_classes)
        )

    def forward(self, images, metadata):
        B = images.size(0)

        # ── Image branch ────────────────────────────────────────────────────
        feat = self.backbone(images)                 # (B, C, H, W)
        feat = feat.flatten(2).permute(0, 2, 1)      # (B, Ri, C)
        img_tokens = self.img_proj(feat)              # (B, Ri, D)

        # ── Metadata branch ─────────────────────────────────────────────────
        meta = self.meta_proj(metadata)               # (B, D*META_TOKENS)
        meta_tokens = meta.view(B, CFG.META_TOKENS, CFG.D)  # (B, Rm, D)

        # ── Bidirectional Cross-Attention ────────────────────────────────────
        for layer in self.cross_attn_layers:
            img_tokens, meta_tokens = layer(img_tokens, meta_tokens)

        # ── Pooling: mean over image spatial tokens ──────────────────────────
        pooled = img_tokens.mean(dim=1)              # (B, D)

        return self.head(pooled)

# ─────────────────────────────────────────────
# 11. TTA  (identical to GNN)
# ─────────────────────────────────────────────
def predict_with_tta(model, image_tensor, metadata_tensor):
    variants = [
        image_tensor,
        TF.hflip(image_tensor),
        TF.vflip(image_tensor),
        TF.rotate(image_tensor,  90),
        TF.rotate(image_tensor, 270),
    ]
    model.eval()
    probs_list = []
    with torch.no_grad():
        for v in variants:
            out  = model(v.to(device), metadata_tensor.to(device))
            prob = torch.softmax(out, dim=1).cpu()
            probs_list.append(prob)
    return torch.stack(probs_list).mean(dim=0)

# ─────────────────────────────────────────────
# 12. Train / Validate  (identical to GNN)
# ─────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scheduler):
    model.train()
    total_loss, correct, total = 0, 0, 0
    optimizer.zero_grad()

    for step, (imgs, metas, labels) in enumerate(
            tqdm(loader, desc="Train", leave=False)):

        imgs, metas, labels = (imgs.to(device),
                               metas.to(device),
                               labels.to(device))
        out  = model(imgs, metas)
        loss = criterion(out, labels) / CFG.GRAD_ACCUM
        loss.backward()

        if (step + 1) % CFG.GRAD_ACCUM == 0 or (step + 1) == len(loader):
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * CFG.GRAD_ACCUM
        correct    += (out.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def valid_one_epoch(model, loader, use_tta=False):
    model.eval()
    preds, trues, probs_all = [], [], []

    with torch.no_grad():
        for imgs, metas, labels in tqdm(loader, desc="Val  ", leave=False):
            if use_tta:
                batch_probs = []
                for i in range(imgs.size(0)):
                    p = predict_with_tta(
                        model,
                        imgs[i].unsqueeze(0),
                        metas[i].unsqueeze(0)
                    )
                    batch_probs.append(p)
                prob = torch.cat(batch_probs, dim=0)
            else:
                imgs, metas = imgs.to(device), metas.to(device)
                out  = model(imgs, metas)
                prob = torch.softmax(out, dim=1).cpu()

            preds.extend(prob.argmax(1).numpy())
            trues.extend(labels.numpy())
            probs_all.extend(prob.numpy())

    trues     = np.array(trues)
    preds     = np.array(preds)
    probs_all = np.array(probs_all)

    acc = accuracy_score(trues, preds)
    f1  = f1_score(trues, preds, average="weighted")
    auc = (roc_auc_score(trues, probs_all[:, 1]) if NUM_CLASSES == 2
           else roc_auc_score(trues, probs_all,
                              multi_class="ovr", average="weighted"))

    return acc, f1, auc, trues, preds, probs_all

# ─────────────────────────────────────────────
# 13. DataLoaders & Model
# ─────────────────────────────────────────────
train_loader = DataLoader(
    SkinDataset(train_df, get_train_transform()),
    batch_size  = CFG.BATCH_SIZE,
    shuffle     = True,
    num_workers = CFG.NUM_WORKERS,
    pin_memory  = True
)
val_loader = DataLoader(
    SkinDataset(val_df, get_val_transform()),
    batch_size  = CFG.BATCH_SIZE,
    shuffle     = False,
    num_workers = CFG.NUM_WORKERS,
    pin_memory  = True
)

model = ConvNeXtBiCrossAttn(META_DIM, NUM_CLASSES).to(device)

trainable = sum(p.numel() for p in model.parameters()
                if p.requires_grad) / 1e6
total_p   = sum(p.numel() for p in model.parameters()) / 1e6
print(f"\nTotal params    : {total_p:.1f}M")
print(f"Trainable params: {trainable:.1f}M  "
      f"(first {CFG.FREEZE_STAGES} stages frozen)")

# ─────────────────────────────────────────────
# 14. Optimizer  (same LR split as GNN)
# ─────────────────────────────────────────────
optimizer = torch.optim.AdamW([
    # Backbone (unfrozen stages)
    {"params": [p for p in model.backbone.parameters()
                if p.requires_grad],
     "lr": CFG.LR,      "weight_decay": CFG.WEIGHT_DECAY},
    # Image projection
    {"params": model.img_proj.parameters(),
     "lr": CFG.HEAD_LR, "weight_decay": CFG.WEIGHT_DECAY},
    # Meta projection
    {"params": model.meta_proj.parameters(),
     "lr": CFG.HEAD_LR, "weight_decay": CFG.WEIGHT_DECAY},
    # Cross-attention layers
    {"params": model.cross_attn_layers.parameters(),
     "lr": CFG.HEAD_LR, "weight_decay": CFG.WEIGHT_DECAY},
    # Classification head
    {"params": model.head.parameters(),
     "lr": CFG.HEAD_LR, "weight_decay": CFG.WEIGHT_DECAY},
])

total_steps = CFG.EPOCHS * len(train_loader)
scheduler   = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr      = [CFG.LR, CFG.HEAD_LR, CFG.HEAD_LR,
                   CFG.HEAD_LR, CFG.HEAD_LR],
    total_steps = total_steps,
    pct_start   = 0.1,
    anneal_strategy = "cos"
)

criterion    = FocalLoss()
best_f1      = 0.0
patience_cnt = 0

# ─────────────────────────────────────────────
# 15. Training Loop
# ─────────────────────────────────────────────
print(f"\n{'='*55}")
print("  ConvNeXt-Tiny + Bidirectional Cross-Attention  —  Training")
print(f"{'='*55}")

for epoch in range(CFG.EPOCHS):
    tr_loss, tr_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, scheduler)
    acc, f1, auc, _, _, _ = valid_one_epoch(
        model, val_loader, use_tta=False)

    print(f"Epoch {epoch+1:02d}/{CFG.EPOCHS} | "
          f"Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.4f} | "
          f"Val Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

    if f1 > best_f1:
        best_f1      = f1
        patience_cnt = 0
        torch.save(model.state_dict(), "best_convnext_bicrossattn.pth")
        print(f"Saved (F1: {best_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= CFG.PATIENCE:
            print(f"  ⏹ Early stopping at epoch {epoch+1}")
            break

print(f"\nBest Val F1: {best_f1:.4f}")

# ─────────────────────────────────────────────
# 16. Final Evaluation on Test Set
# ─────────────────────────────────────────────
print(f"\n{'='*55}")
print("  Final Evaluation on TEST SET")
print(f"{'='*55}")

model.load_state_dict(
    torch.load("best_convnext_bicrossattn.pth", map_location=device))

test_loader = DataLoader(
    SkinDataset(test_df, get_val_transform()),
    batch_size  = CFG.BATCH_SIZE,
    shuffle     = False,
    num_workers = CFG.NUM_WORKERS,
    pin_memory  = True
)

# Without TTA
acc, f1, auc, trues, preds, _ = valid_one_epoch(
    model, test_loader, use_tta=False)
print(f"\n[Without TTA] Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

# With TTA
acc_t, f1_t, auc_t, trues, preds_tta, _ = valid_one_epoch(
    model, test_loader, use_tta=True)
print(f"[With    TTA] Acc: {acc_t:.4f} | F1: {f1_t:.4f} | AUC: {auc_t:.4f}")

print("\n" + "="*55)
print("FINAL RESULTS — ConvNeXt-Tiny + BiCrossAttn  (with TTA)")
print("="*55)
print(classification_report(trues, preds_tta, target_names=classes_str))

# ✅ Save for visualization notebook
np.save("convnext_trues.npy",     trues)
np.save("convnext_preds_tta.npy", preds_tta)
print("\n[✓] Saved convnext_trues.npy & convnext_preds_tta.npy for visualization")

Using device: cuda
Classes     : ['0', '1']
Num classes : 2
Train size  : 11941
Val size    : 2559
Test size   : 2559

Train distribution:
label
1    5971
0    5970
Name: count, dtype: int64

Meta columns (29): ['age_scaled', 'melanocytic', 'sex_Unknown', 'sex_female', 'sex_male', 'anatom_site_general_Unknown', 'anatom_site_general_anterior torso', 'anatom_site_general_head/neck', 'anatom_site_general_lateral torso', 'anatom_site_general_lower extremity', 'anatom_site_general_oral/genital', 'anatom_site_general_palms/soles', 'anatom_site_general_posterior torso', 'anatom_site_general_upper extremity', 'dermoscopic_type_Unknown', 'dermoscopic_type_contact non-polarized', 'dermoscopic_type_contact polarized', 'dermoscopic_type_non-contact polarized', 'diagnosis_confirm_type_Unknown', 'diagnosis_confirm_type_confocal microscopy with consensus dermoscopy', 'diagnosis_confirm_type_histopathology', 'diagnosis_confirm_type_serial imaging showing no change', 'diagnosis_confirm_type_single imag

Epoch 01/25 | Loss: 0.0356 | Train Acc: 0.7143 | Val Acc: 0.8034 | F1: 0.8030 | AUC: 0.8945
Saved (F1: 0.8030)


Epoch 02/25 | Loss: 0.0276 | Train Acc: 0.8100 | Val Acc: 0.7835 | F1: 0.7743 | AUC: 0.9237


Epoch 03/25 | Loss: 0.0248 | Train Acc: 0.8347 | Val Acc: 0.8441 | F1: 0.8438 | AUC: 0.9384
Saved (F1: 0.8438)


KeyboardInterrupt: 

In [ ]:
 ═══════════════════════════════════════════
#  VISUALIZATION 1 — Confusion Matrix
# ═══════════════════════════════════════════
cm = confusion_matrix(trues, preds_tta)
 
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=classes_str, yticklabels=classes_str, ax=ax,
            linewidths=0.5, linecolor="white",
            annot_kws={"size": 13, "weight": "bold"})
ax.set_ylabel("True Label",      fontsize=11)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_title("Confusion Matrix — ConvNeXt-Tiny + BiCrossAttn",
             fontsize=12, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()
print("✅  Confusion Matrix saved")
 
# ═══════════════════════════════════════════
#  VISUALIZATION 2 — ROC Curve
# ═══════════════════════════════════════════
if NUM_CLASSES == 2:
    fpr, tpr, thresholds = roc_curve(trues, probs_pos)
    roc_auc              = sk_auc(fpr, tpr)
    youden_idx           = np.argmax(tpr - fpr)
    opt_thr              = thresholds[youden_idx]
 
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, lw=2, color="purple", label=f"AUC = {roc_auc:.4f}")
    ax.scatter(fpr[youden_idx], tpr[youden_idx], color="red", zorder=5, s=80,
               label=f"Optimal thr = {opt_thr:.3f}")
    ax.fill_between(fpr, tpr, alpha=0.08, color="purple")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate", fontsize=11)
    ax.set_ylabel("True Positive Rate",  fontsize=11)
    ax.set_title("ROC Curve — ConvNeXt-Tiny + BiCrossAttn",
                 fontsize=12, fontweight="bold", pad=12)
    ax.legend(fontsize=10); ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.OUTPUT_DIR, "roc_curve.png"), dpi=150)
    plt.show()
    print("✅  ROC Curve saved")
else:
    # One-vs-Rest ROC for multiclass
    from sklearn.preprocessing import label_binarize
    trues_bin = label_binarize(trues, classes=list(range(NUM_CLASSES)))
    fig, ax   = plt.subplots(figsize=(7, 5))
    colors    = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
    for i, (cls, col) in enumerate(zip(classes_str, colors)):
        fpr_i, tpr_i, _ = roc_curve(trues_bin[:, i], probs_all[:, i])
        auc_i = sk_auc(fpr_i, tpr_i)
        ax.plot(fpr_i, tpr_i, lw=2, color=col, label=f"{cls} (AUC={auc_i:.3f})")
    ax.plot([0,1],[0,1],"k--", lw=1)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title("One-vs-Rest ROC — ConvNeXt-Tiny + BiCrossAttn",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.OUTPUT_DIR, "roc_curve_ovr.png"), dpi=150)
    plt.show()
    print("✅  ROC Curve (OvR) saved")
 
# ═══════════════════════════════════════════
#  VISUALIZATION 3 — Training Curves
#  Reads history dict saved during training.
#  If not found, prints a clear instruction.
# ═══════════════════════════════════════════
HISTORY_PATH = "convnext_history.npy"
 
try:
    history = np.load(HISTORY_PATH, allow_pickle=True).item()
 
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
 
    axes[0].plot(history["train_loss"], label="Train", marker="o", ms=3, color="steelblue")
    axes[0].plot(history["val_loss"],   label="Val",   marker="o", ms=3, color="orange")
    axes[0].set_title("Loss");       axes[0].legend(); axes[0].grid(True, alpha=0.4)
 
    axes[1].plot(history["train_auc"],   label="Train",   alpha=0.6, color="steelblue")
    axes[1].plot(history["val_auc"],     label="Val",     alpha=0.6, color="orange")
    if "val_auc_ema" in history:
        axes[1].plot(history["val_auc_ema"], label="Val EMA", lw=2, color="red")
    axes[1].set_title("AUC-ROC");    axes[1].legend(); axes[1].grid(True, alpha=0.4)
 
    axes[2].plot(history["val_f1"], label="Val F1", color="green", marker="o", ms=3)
    axes[2].set_title("Val F1");     axes[2].legend(); axes[2].grid(True, alpha=0.4)
 
    if "threshold" in history:
        axes[3].plot(history["threshold"], label="Optimal Threshold",
                     color="purple", marker="o", ms=3)
        axes[3].axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Default 0.5")
        axes[3].set_title("Optimal Threshold")
    else:
        axes[3].plot(history.get("val_acc", []), label="Val Acc",
                     color="purple", marker="o", ms=3)
        axes[3].set_title("Val Accuracy")
    axes[3].legend(); axes[3].grid(True, alpha=0.4)
 
    plt.suptitle("ConvNeXt-Tiny + BiCrossAttn — Training Curves",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.OUTPUT_DIR, "training_curves.png"), dpi=150)
    plt.show()
    print("✅  Training Curves saved")
 
except FileNotFoundError:
    print(f"⚠️  '{HISTORY_PATH}' not found.")
    print("   ➜ In your training loop, after each epoch add:")
    print("       history['train_loss'].append(tr_loss)")
    print("       history['val_loss'].append(val_loss)")
    print("       history['train_auc'].append(tr_auc)")
    print("       history['val_auc'].append(val_auc)")
    print("       history['val_f1'].append(f1)")
    print("   Then at the end: np.save('convnext_history.npy', history)")
 
# ═══════════════════════════════════════════
#  VISUALIZATION 4 — Calibration Curve
# ═══════════════════════════════════════════
fig, ax = plt.subplots(figsize=(6, 5))
 
if NUM_CLASSES == 2:
    fraction_of_pos, mean_pred = calibration_curve(
        trues, probs_pos, n_bins=10, strategy="uniform")
    ax.plot(mean_pred, fraction_of_pos, marker="o", lw=2,
            color="purple", label="ConvNeXt BiCrossAttn")
else:
    # Average calibration across all classes
    from sklearn.preprocessing import label_binarize
    trues_bin = label_binarize(trues, classes=list(range(NUM_CLASSES)))
    for i, cls in enumerate(classes_str):
        fp, mp = calibration_curve(trues_bin[:, i], probs_all[:, i],
                                   n_bins=10, strategy="uniform")
        ax.plot(mp, fp, marker="o", lw=1.5, alpha=0.7, label=f"Class {cls}")
 
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
ax.set_xlabel("Mean Predicted Probability", fontsize=11)
ax.set_ylabel("True Fraction of Positives",  fontsize=11)
ax.set_title("Calibration Curve — ConvNeXt-Tiny + BiCrossAttn",
             fontsize=12, fontweight="bold", pad=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, "calibration_curve.png"), dpi=150)
plt.show()
print("✅  Calibration Curve saved")
 
# ─────────────────────────────────────────────
print(f"\n✅  All visualizations saved to: {CFG.OUTPUT_DIR}")